# 07_specifications: What Is Required?

**Lab report:** Invariant Specification Demystifies Translation

Notebook 00 asked whether GLOSSOPETRAE exposes a meaningful object called a *specification*.

This notebook asks the next question:

> What exactly is specified?

The goal is to move from:

```text
Specification → Language
```

to a more explicit map:

```text
Specification =
{
  grammar,
  lexicon,
  phonology,
  writing system,
  translation relations,
  evolution rules,
  ...
}
```

## 1. Setup

Clone or reuse the upstream GLOSSOPETRAE repository.

In [ ]:
from pathlib import Path
import os
import re
import json
import subprocess
from collections import Counter, defaultdict

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Find candidate specification files

Search paths and filenames for the concepts most likely to encode a language specification.

In [ ]:
keywords = [
    "spec", "schema", "config", "language", "grammar",
    "lexicon", "phonology", "phoneme", "sound",
    "script", "glyph", "writing", "translation",
    "evolution", "daughter", "descendant", "lineage"
]

candidate_files = []
for root, dirs, files in os.walk(UPSTREAM_DIR):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__", ".venv", "node_modules"}]
    for f in files:
        path = Path(root) / f
        rel = path.relative_to(UPSTREAM_DIR)
        lower = str(rel).lower()
        hits = [k for k in keywords if k in lower]
        if hits:
            candidate_files.append({
                "path": str(rel),
                "suffix": path.suffix,
                "hits": ", ".join(hits)
            })

len(candidate_files), candidate_files[:10]

In [ ]:
try:
    import pandas as pd
    candidates_df = pd.DataFrame(candidate_files).sort_values(["hits", "path"])
    display(candidates_df.head(100))
    candidates_df.to_csv(DATA / "07_candidate_spec_files.csv", index=False)
except Exception:
    for row in candidate_files[:100]:
        print(row)

## 3. Inspect text for specification vocabulary

Count how often candidate terms appear in source, docs, configs, and examples.

In [ ]:
TEXT_SUFFIXES = {".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml", ".js", ".ts", ".html", ".css"}

content_rows = []
for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue
    lower = text.lower()
    for k in keywords:
        count = lower.count(k)
        if count:
            content_rows.append({
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "keyword": k,
                "count": count
            })

try:
    import pandas as pd
    content_df = pd.DataFrame(content_rows)
    keyword_summary = content_df.groupby("keyword", as_index=False)["count"].sum().sort_values("count", ascending=False)
    display(keyword_summary)
    keyword_summary.to_csv(DATA / "07_keyword_summary.csv", index=False)
except Exception:
    print(Counter([r["keyword"] for r in content_rows]))

## 4. Extract nearby snippets

Look at nearby text around high-value terms. This helps distinguish names from actual definitions.

In [ ]:
def snippets_for(term, max_snippets=20, radius=120):
    rows = []
    for path in UPSTREAM_DIR.rglob("*"):
        if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
            continue
        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue
        lower = text.lower()
        idx = lower.find(term.lower())
        if idx >= 0:
            start = max(0, idx - radius)
            end = min(len(text), idx + len(term) + radius)
            snippet = text[start:end].replace("\n", " ")
            rows.append({
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "term": term,
                "snippet": snippet
            })
            if len(rows) >= max_snippets:
                break
    return rows

snippet_terms = ["specification", "language", "grammar", "phonology", "script", "translation", "evolution"]
snippets = []
for term in snippet_terms:
    snippets.extend(snippets_for(term, max_snippets=12))

try:
    import pandas as pd
    snippets_df = pd.DataFrame(snippets)
    display(snippets_df)
    snippets_df.to_csv(DATA / "07_snippets.csv", index=False)
except Exception:
    for s in snippets:
        print("\n---", s["term"], s["path"])
        print(s["snippet"])

## 5. Component checklist

Create a working checklist of specification components. Mark whether each component appears required, optional, generated, or unresolved.

This table should be edited after inspecting the upstream examples.

In [ ]:
component_rows = [
    {
        "component": "lexicon",
        "question": "Are words or roots specified directly?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "grammar",
        "question": "Are syntax or morphology rules specified?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "phonology",
        "question": "Are phonemes, syllables, or sound constraints specified?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "orthography / script",
        "question": "Are writing rules or glyph systems specified or generated?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "semantics",
        "question": "Are meanings, concepts, or semantic domains specified?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "translation mapping",
        "question": "Are cross-language correspondences specified or generated?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
    {
        "component": "evolution rules",
        "question": "Are sound change or lineage rules specified?",
        "status": "unresolved",
        "evidence_path": "",
        "notes": ""
    },
]

try:
    import pandas as pd
    components_df = pd.DataFrame(component_rows)
    display(components_df)
    components_df.to_csv(DATA / "07_component_checklist.csv", index=False)
except Exception:
    for row in component_rows:
        print(row)

## 6. Minimal specification search

Look for small config, example, or demo files that could serve as a minimal specification.

In [ ]:
example_like = []
for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir():
        continue
    rel = path.relative_to(UPSTREAM_DIR)
    lower = str(rel).lower()
    if any(k in lower for k in ["example", "demo", "sample", "config", "language"]) and path.suffix.lower() in TEXT_SUFFIXES:
        try:
            size = path.stat().st_size
        except Exception:
            size = None
        example_like.append({
            "path": str(rel),
            "suffix": path.suffix,
            "size_bytes": size
        })

example_like = sorted(example_like, key=lambda r: (r["size_bytes"] if r["size_bytes"] is not None else 10**12, r["path"]))

try:
    import pandas as pd
    example_df = pd.DataFrame(example_like)
    display(example_df.head(50))
    example_df.to_csv(DATA / "07_minimal_spec_candidates.csv", index=False)
except Exception:
    for row in example_like[:50]:
        print(row)

## 7. Required vs optional

After identifying the smallest runnable example, classify each component.

The point is not to prove the whole system; it is to understand what the engine must know before it can produce a language.

In [ ]:
required_optional_template = [
    {"component": "lexicon", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
    {"component": "grammar", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
    {"component": "phonology", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
    {"component": "script", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
    {"component": "semantics", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
    {"component": "evolution rules", "present_in_minimal_example": None, "required": None, "generated_if_missing": None},
]

try:
    import pandas as pd
    req_df = pd.DataFrame(required_optional_template)
    display(req_df)
    req_df.to_csv(DATA / "07_required_optional_template.csv", index=False)
except Exception:
    for row in required_optional_template:
        print(row)

## 8. Specification diagram v2

Notebook 00 drew specification as a single node. Notebook 07 expands it into component constraints.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.axis("off")

nodes = {
    "Specification": (0.5, 0.90),
    "Grammar": (0.18, 0.70),
    "Lexicon": (0.34, 0.70),
    "Phonology": (0.50, 0.70),
    "Script": (0.66, 0.70),
    "Evolution\nRules": (0.84, 0.70),
    "Language": (0.5, 0.48),
    "Speech": (0.27, 0.28),
    "Text": (0.50, 0.28),
    "Glyphs": (0.73, 0.28),
    "Translation +\nDescendants": (0.5, 0.10),
}

for label, (x, y) in nodes.items():
    fontsize = 12 if label == "Specification" else 10
    ax.text(
        x, y, label,
        ha="center", va="center", fontsize=fontsize,
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2)
    )

edges = [
    ("Specification", "Grammar"),
    ("Specification", "Lexicon"),
    ("Specification", "Phonology"),
    ("Specification", "Script"),
    ("Specification", "Evolution\nRules"),
    ("Grammar", "Language"),
    ("Lexicon", "Language"),
    ("Phonology", "Language"),
    ("Script", "Language"),
    ("Evolution\nRules", "Language"),
    ("Language", "Speech"),
    ("Language", "Text"),
    ("Language", "Glyphs"),
    ("Speech", "Translation +\nDescendants"),
    ("Text", "Translation +\nDescendants"),
    ("Glyphs", "Translation +\nDescendants"),
    ("Language", "Translation +\nDescendants"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

path = FIGURES / "07_specification_components.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 9. Hypothesis 1

**Hypothesis 1**

A linguistic specification is a collection of constraints governing sound, form, meaning, writing, translation, and change.

Different representations can be generated from the same specification. Translation and language evolution should therefore be evaluated by how much correspondence they preserve, not merely by surface word matching.

## 10. Questions for Notebook 13

Notebook 13 should move from specification contents to transformation:

| Question | Motivation |
|---|---|
| What survives translation? | Tests the core report claim |
| What changes under translation? | Separates representation from specification |
| Which correspondences persist? | Defines a measurable invariant |
| Can we compare original and translated structures? | Turns the claim into an experiment |